In [1]:
import pandas as pd
import re
from ollama import chat, ChatResponse
from tqdm import tqdm  # 用於顯示進度條

In [2]:
# --- 設定 ---
FILE_PATH = './data/c3_d_test.xlsx'     # 資料集路徑

In [3]:
# --- 函數：呼叫 Ollama (使用官方庫) ---
def query_ollama(prompt, model='gemma3:12b', context_len=1536, think=False):
    try:
        # 使用 ollama.chat 直接呼叫
        response: ChatResponse = chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            # 您可以在此加入 options，例如 temperature
            options={
                'num_ctx': context_len,  # 在這裡指定 Context Length
                'temperature': 0.0  # 讓輸出結果穩定
            },
            think=think  # 設定為 True 以獲取推理欄位
        )
        # 回傳模型的內容
        return response.message.content
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""

In [4]:
# --- 函數：解析模型輸出 ---
def extract_answer(text):
    """
    從模型的回答中提取選項數字 (1, 2, 3, 4)。
    DeepSeek R1 會輸出 <think>...</think> 思考過程，需要過濾。
    """
    if not text:
        return None

    # 1. 移除 <think> 標籤內的內容 (DeepSeek R1 特有)
    # flags=re.DOTALL 確保 . 可以匹配換行符號
    text_cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. 策略 A: 尋找明確的 "Answer: X" 格式 (Prompt 中有要求)
    match = re.search(r'Answer:\s*([1-4])', text_cleaned, re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # 3. 策略 B: 如果沒有標準格式，尋找最後出現的數字 1-4
    # 這是一個備用方案
    numbers = re.findall(r'\b([1-4])\b', text_cleaned)
    if numbers:
        return int(numbers[-1])
        
    return None  # 解析失敗

In [5]:
# --- 主程式 ---
# 1. 讀取資料
print(f"正在讀取 {FILE_PATH}...")
try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    print(f"讀取檔案失敗: {e}")

# 檢查必要的欄位
required_columns = ['ID', 'Article', 'Question', 'Option1', 'Option2', 'Option3', 'Option4', 'Answer']
if not all(col in df.columns for col in required_columns):
    print(f"錯誤: Excel 檔案缺少必要的欄位。需要: {required_columns}")

正在讀取 ./data/c3_d_test.xlsx...


In [6]:
df

,ID,Article,Question,Option1,Option2,Option3,Option4,Answer
0,1,男：你这么生气，是不是跟你的那位先生吵架了?\n女：要不是他，我会这么生气吗?,女的到底生谁的气?,老师,学生,丈夫,孩子,3
1,2,男：就要放暑假了，你是回家还是出去旅游？\n女：我很想旅游，但是妈妈打电话来说要我在学校看书...,女的意思是什么？,要回家,要去旅游,留在学校看书,不想考研究生,3
2,3,男：还是古典音乐好听，我就受不了摇滚乐，实在太吵了。\n女：那是因为你没听过现场，流行乐和摇...,男的喜欢什么样的音乐?,古典音乐,摇滚音乐,流行音乐,乡村音乐,1
3,4,男：最近怎么腰酸背痛的?\n女：你当你年轻呢，悠着点儿吧，别玩命干了。,女的是提醒男的什么?,注意身体,注意营养,注意锻炼,注意病情,1
4,5,女：一回头发现你不见了，怎么突然停下来了?\n男：没办法，手上拿的袋子实在太重了。,男的为什么停下来?,袋子太重,看不见路,不认识路,女的不见了,1
...,...,...,...,...,...,...,...,...
1885,1886,女：天哪!你总算到了!是不是路上堵车了?\n男：没堵车，是我的手表不知道为什么，慢了半个小时。,男的为什么会迟到?,堵车了,生病了,手表坏了,手表慢了,4
1886,1887,男：中秋节有什么打算?\n女：我想和妹妹去丽江。\n男：丽江?我去年去过那儿，那儿风景很美，...,男的认为丽江怎么样?,风景好,很干燥,污染严重,经济增长快,1
1887,1888,男：您好，我想去北京大学，该在哪一站下车?\n女：下一站下车，换307路，到中关园站下车就到了。,男的要去哪儿?,火车站,中关村,下一站,北京大学,4
1888,1889,女：昨天下午下班后我去办公室找你，你怎么不在?\n男：我去饭馆了，中午吃饭的时候我把手机忘在...,男的去饭馆做什么?,吃饭,买饭,找手机,找手表,3


In [7]:
# --- 主程式 ---
def main():
    correct_count = 0
    total_questions = len(df)
    results = []

    print(f"開始使用 {MODEL_NAME} 處理 {total_questions} 個問題...")

    # 2. 遍歷每一列進行預測
    for index, row in tqdm(df.iterrows(), total=total_questions):
        article = row['Article']
        question = row['Question']
        # 建立選項字典方便取用
        options = {
            1: row['Option1'],
            2: row['Option2'],
            3: row['Option3'],
            4: row['Option4']
        }
        actual_answer = row['Answer']
        #print(actual_answer)

        # 1. 過濾掉 NaN 或空值的選項 (假設 options 是 dict 或 list)
        # 這裡用 str(v) != 'nan' 是為了處理 pandas 的情況
        valid_options = {k: v for k, v in options.items() if v and str(v).lower() != 'nan'}
        
        # 2. 動態生成選項字串
        options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
        valid_numbers = ", ".join([str(k) for k in valid_options.keys()])

        # 建構 Prompt
        # 這裡的設計是為了讓 DeepSeek R1 能夠先思考再回答，並最後輸出固定格式
        prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
Analyze the article and select the best option from the available choices listed above ({valid_numbers}). 
If an option is missing or empty, do not select it.

Finally, output the answer in the exact format: "Answer: X", where X is the option number.
Do not output anything else after the answer.
"""
        
        # 呼叫模型
        response_text = query_ollama(prompt, model=MODEL_NAME, think=THINK_MODE)
        
        # 解析答案
        predicted_answer = extract_answer(response_text)
        
        # 比對結果
        is_correct = False
        try:
            # 處理 Excel 讀入的答案可能是浮點數或字串的情況
            actual_answer_int = int(actual_answer)
            if predicted_answer is not None and predicted_answer == actual_answer_int:
                correct_count += 1
                is_correct = True
            else:
                print("Article:", article)
                print("Question:", question)
                print("Option:", "1.", row['Option1'], "2.", row['Option2'], "3.", row['Option3'], "4.", row['Option4'])
                print("Answer:", actual_answer)
                print("Predicted Answer:", predicted_answer)
                print('-'*30)
        except ValueError:
            pass # 忽略無效的標準答案格式

        # 儲存詳細結果
        results.append({
            "ID": row['ID'],
            "Actual": actual_answer,
            "Predicted": predicted_answer,
            "Correct": is_correct,
            "Model_Response": response_text
        })

        #print(predicted_answer)
        if (index+1) % 500 == 0:
            print(correct_count, " / ", (index+1), " = ", 100.0*correct_count/(index+1), "%")
            

    # 3. 計算並輸出正確率
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0
    print("\n" + "="*30)
    print(f"處理完成！")
    print(f"總題數: {total_questions}")
    print(f"答對題數: {correct_count}")
    print(f"正確率 (Accuracy): {accuracy:.2f}%")
    print("="*30)

    # 4. 將結果存檔
    output_file = "result_analysis.xlsx"
    pd.DataFrame(results).to_excel(output_file, index=False)
    print(f"詳細結果已儲存至 {output_file}")

In [8]:
MODEL_NAME = 'gemma3:12b'  # 模型名稱
THINK_MODE = False
main()

開始使用 gemma3:12b 處理 1890 個問題...


  1%|█▏                                                                              | 28/1890 [00:17<19:09,  1.62it/s]

Article: 女：正点应该是12:30到站，现在已经12:40了，火车怎么还不来?
男：听说晚点20分钟，再等10分钟吧。
Question: 火车几点到站?
Option: 1. 12:30 2. 12:40 3. 12:50 4. 13:00
Answer: 3
Predicted Answer: 1
------------------------------


  2%|█▍                                                                              | 33/1890 [00:20<18:18,  1.69it/s]

Article: 女：明天的会议很重要，经理要求我们八点半以前就要到公司，你没问题吧？
男：您放心，我一定不会迟到的。
Question: 为了不迟到，他可能几点到公司？
Option: 1. 8:20 2. 8:40 3. 8:50 4. nan
Answer: 1
Predicted Answer: 2
------------------------------


...

Article: 男：这趟公交车怎么还不来啊?
女：是啊，现在都六点半了，我还得回家做晚饭呢。
男：真让人着急，我都等了20分钟了。
女：是啊，也不知道是怎么回事。
Question: 男的从几点开始等车?
Option: 1. 6:10 2. 6:20 3. 6:30 4. 6:50
Answer: 1
Predicted Answer: 2
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▉| 1889/1890 [20:26<00:00,  1.70it/s]

Article: 女：昨天下午下班后我去办公室找你，你怎么不在?
男：我去饭馆了，中午吃饭的时候我把手机忘在那儿了。
Question: 男的去饭馆做什么?
Option: 1. 吃饭 2. 买饭 3. 找手机 4. 找手表
Answer: 3
Predicted Answer: 1
------------------------------


100%|██████████████████████████████████████████████████████████████████████████████| 1890/1890 [20:26<00:00,  1.54it/s]


處理完成！
總題數: 1890
答對題數: 1732
正確率 (Accuracy): 91.64%
詳細結果已儲存至 result_analysis.xlsx


In [9]:
MODEL_NAME = 'qwen3:8b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:8b 處理 1890 個問題...


  2%|█▋                                                                              | 41/1890 [00:13<08:36,  3.58it/s]

Article: 女：自己多注意，到了就给家里打个电话。
男：记住了。飞机马上要飞了，我关手机了。
Question: 男的在什么地方?
Option: 1. 汽车上 2. 飞机上 3. 飞机场 4. nan
Answer: 2
Predicted Answer: 3
------------------------------


  3%|██▌                                                                             | 60/1890 [00:18<08:44,  3.49it/s]

Article: 男：天气真热啊!
女：今天是36度，明天会下一场小雨，温度下降三度。
男：后天呢?后天我有一场足球比赛。
女：后天比明天又要高两度。
Question: 后天的气温是多少度?
Option: 1. 36°C 2. 33°C 3. 32°C 4. 35°C
Answer: 4
Predicted Answer: 3
------------------------------


...

Article: 男：妈，我放在冰箱里的巧克力呢?
女：噢，昨天邻居家的小明过来，我就先拿给她吃了，以后妈再给你买。
Question: 从对话中可以知道什么?
Option: 1. 小明拿了男的的巧克力 2. 妈妈给男的买了巧克力 3. 小明是他们家的邻居 4. 小明妈妈要买巧克力
Answer: 3
Predicted Answer: 1
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▉| 1889/1890 [08:52<00:00,  3.61it/s]

Article: 女：昨天下午下班后我去办公室找你，你怎么不在?
男：我去饭馆了，中午吃饭的时候我把手机忘在那儿了。
Question: 男的去饭馆做什么?
Option: 1. 吃饭 2. 买饭 3. 找手机 4. 找手表
Answer: 3
Predicted Answer: 1
------------------------------


100%|██████████████████████████████████████████████████████████████████████████████| 1890/1890 [08:52<00:00,  3.55it/s]


處理完成！
總題數: 1890
答對題數: 1759
正確率 (Accuracy): 93.07%
詳細結果已儲存至 result_analysis.xlsx


In [10]:
MODEL_NAME = 'qwen2.5:7b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen2.5:7b 處理 1890 個問題...


  3%|██▎                                                                             | 55/1890 [00:17<08:03,  3.79it/s]

Article: 女：这是我给咱们儿子新买的玩具汽车。
男：他不是已经有一个了吗?
女：别提了，那个已经被弄坏了。
男：这个看起来比上一个好呢。
Question: 他们是什么关系?
Option: 1. 恋人 2. 夫妻 3. 父母与孩子 4. nan
Answer: 2
Predicted Answer: 3
------------------------------


  3%|██▌                                                                             | 60/1890 [00:19<09:00,  3.39it/s]

Article: 男：天气真热啊!
女：今天是36度，明天会下一场小雨，温度下降三度。
男：后天呢?后天我有一场足球比赛。
女：后天比明天又要高两度。
Question: 后天的气温是多少度?
Option: 1. 36°C 2. 33°C 3. 32°C 4. 35°C
Answer: 4
Predicted Answer: 3
------------------------------


...

Article: 男：警察同志，我真的不是故意的，您就高抬贵手放了我吧。我妻子还在等我去接她呢。
女：别人都看到红灯了，怎么就你看不见呢。你的驾照?
男：啊，对不起，我太着急了，驾照也忘了带了。
女：那就麻烦你跟我走一趟吧。
Question: 男的发生了什么事?
Option: 1. 他在等他妻子 2. 他没看见红灯 3. 他没带照相机 4. 他违反了交通规则
Answer: 4
Predicted Answer: 2
------------------------------


 99%|█████████████████████████████████████████████████████████████████████████████▍| 1876/1890 [08:58<00:04,  3.41it/s]

Article: 男：这趟公交车怎么还不来啊?
女：是啊，现在都六点半了，我还得回家做晚饭呢。
男：真让人着急，我都等了20分钟了。
女：是啊，也不知道是怎么回事。
Question: 男的从几点开始等车?
Option: 1. 6:10 2. 6:20 3. 6:30 4. 6:50
Answer: 1
Predicted Answer: 3
------------------------------


100%|██████████████████████████████████████████████████████████████████████████████| 1890/1890 [09:02<00:00,  3.49it/s]



處理完成！
總題數: 1890
答對題數: 1744
正確率 (Accuracy): 92.28%
詳細結果已儲存至 result_analysis.xlsx


In [11]:
MODEL_NAME = 'qwen3:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:14b 處理 1890 個問題...


  2%|█▋                                                                              | 41/1890 [00:17<11:22,  2.71it/s]

Article: 女：自己多注意，到了就给家里打个电话。
男：记住了。飞机马上要飞了，我关手机了。
Question: 男的在什么地方?
Option: 1. 汽车上 2. 飞机上 3. 飞机场 4. nan
Answer: 2
Predicted Answer: 3
------------------------------


  5%|████▏                                                                          | 100/1890 [00:38<11:21,  2.63it/s]

Article: 女：欢迎“童话大王”郑渊洁。您好，恭喜您今年晋升为中国作家首富，去年汶川大地震，您是作家中捐得最多的。您如何看待作家在特殊情况下的捐款行为？
男：其实不在于捐钱多少，只要有这个心就行了，有力出力，有钱出钱。以前得过好多别的奖，都没有幸福感，只是高兴，只有成就感。去年十二月，国家给了我一个“中华慈善奖”。我当时有一个感觉：领这个奖感觉非常好。我觉得一个人要想真正获得幸福，只有一个渠道，就是帮助别人。你住着别墅，开着宝马，你也不幸福，这是我的体会。
女：有没有计划将来办一个慈善机构，在更广泛的范围内帮助人？
男：目前有两种基金会。一种是封闭式的，我只用自己的钱，不要别人捐钱，我要干这种。一种是开放的，用别人的，自己不掏，靠自己的名气。这个也好。
女：《皮皮鲁总动员》近两年的销售情况都非常好，仅二零零九年二月，就销售出一百多万册。您觉得自己作品热销的原因是什么？
男：我觉得可能和两个原因有关，一个是经济危机，一个是甲流，很多孩子周末和暑假原本是要出去玩儿的，包括我的孩子，后来都取消了。不出去以后，很多家长就去书店买书，我估计可能跟这个有关系。
女：您能给我们谈谈自己是怎样成为富人的吗？您的财富观是什么样的呢？
男：我爸是山西人，我妈是浙江人，他们的结合就是钱庄和票号的结合，我的血液里就有理财的东西。对作家来说财富其实是两笔，一笔是稿费，一笔是作品。作品是无形资产，比稿费厉害得多，是真正的摇钱树。这个摇钱树将来还可以派生出很多产品，影视、网络之类。说到稿费这一块儿，我觉得我可能名声不好，给人的感觉是老跟出版社谈钱。我觉得，作家还是要维护自己的利益。如果自己的衣食住行都解决了，就要尽可能地帮助别人，做慈善事业。
Question: 如果设立基金会，男的希望怎么样？
Option: 1. 用自己的钱 2. 呼吁更多人捐款 3. 同时做两种基金会 4. 充分利用自己的知名度
Answer: 1
Predicted Answer: 3
------------------------------


...

Article: 男：妈，我放在冰箱里的巧克力呢?
女：噢，昨天邻居家的小明过来，我就先拿给她吃了，以后妈再给你买。
Question: 从对话中可以知道什么?
Option: 1. 小明拿了男的的巧克力 2. 妈妈给男的买了巧克力 3. 小明是他们家的邻居 4. 小明妈妈要买巧克力
Answer: 3
Predicted Answer: 1
------------------------------


 97%|███████████████████████████████████████████████████████████████████████████▉  | 1840/1890 [10:51<00:17,  2.90it/s]

Article: 女：这次旅行你要是不去，大家都会觉得没意思的。
男：我要是没事能不去吗?
Question: 男的是什么意思?
Option: 1. 男的不想去 2. 男的没有事 3. 男的也想去 4. 大家都有事
Answer: 3
Predicted Answer: 1
------------------------------


100%|██████████████████████████████████████████████████████████████████████████████| 1890/1890 [11:09<00:00,  2.82it/s]


處理完成！
總題數: 1890
答對題數: 1794
正確率 (Accuracy): 94.92%
詳細結果已儲存至 result_analysis.xlsx


In [12]:
MODEL_NAME = 'deepseek-r1:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 deepseek-r1:14b 處理 1890 個問題...


  1%|▉                                                                               | 22/1890 [00:37<46:32,  1.50s/it]

Article: 男：李大姐如果要挤车，您挤得过别人吗?
女：我不挤车，就是说我不跟人家挤，我让人家先上，我最后上，因为我是售票员，我卡门卡得好。
男：那是售票员的技术之一，就是不管多挤，你最后都要能上去。
女：那也是一种技能。
男：您给我们讲一讲，大家坐车都用得上，怎么挤呀?
女：我刚参加工作的时候，不会挤车，因为前门上，后门下，那个真的不会挤，那会儿离开师傅，第一天上班，还特别高兴。结果上班后，早高峰那时候车上人特多，我还特热情地下去，卖完票，人下去了，完了就上不来了。我上不来就影响开车。
男：卖票的上不来?
女：我上不去，那车就不能走啊，后来急得我都要掉眼泪了，我就跟后边那个男同志说，您先下来，我上去吧。人家不同意。完了那司机师傅皱着眉头，叫我过去，从司机那儿爬上去。回家之后，我就想，售票员还有一种技能是必须要学的，就是要学会最后能够卡门上车，你保证自己的安全，还要保证乘客的安全。我记得跟那些老师傅学的时候，有将近两个月的时间，浑身夹得青一块儿紫一块儿的，连上床都上不去。你们可能有的时候坐车一看，都看的是面上的东西，看到一个好的售票员，看不到他奉献了多少业余时间。他要去积累，要跟人打交道，要猜测不同人的心理。
男：得看人特准吧?
女：对，反正我们看那些逃票的，特别准。
Question: 女的怎么跟别人挤车?
Option: 1. 用力挤 2. 看准时机 3. 让别人先上 4. 用手拉别人
Answer: 3
Predicted Answer: 2
------------------------------


  2%|█▎                                                                              | 30/1890 [00:49<49:22,  1.59s/it]

Article: 男：你理发了?我都认不出你了。
女：是啊，昨天理的，我还是觉得短发适合我。
Question: 女的怎么了?
Option: 1. 以前是短发 2. 以前是长发 3. 今天理的发 4. 长发适合她
Answer: 2
Predicted Answer: 3
------------------------------


...

Article: 男：其实价钱不是问题，可是买回来也没用啊。
女：是吗?要是便宜，没用你也能买。
Question: 女的对男的说的话表示什么?
Option: 1. 赞成 2. 欣赏 3. 理解 4. 反对
Answer: 4
Predicted Answer: 3
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▋| 1882/1890 [40:34<00:13,  1.67s/it]

Article: 主持人：今天请来了山东工艺美术学院副院长、青年学者潘鲁生教授。潘教授从事民间艺术研究近20年，提出了“民间手工文化生态保护”的学术命题，在全国范围内产生了很大的影响。潘教授，是什么样的背景促使您提出这个命题的?
潘教授：这是民间艺术研究的一个必然结果。起初我们对民艺的研究也是从民间剪纸、年画、刺绣开始的。后来随着研究的不断深化，民艺作品背后的文化背景、生活背景开始浮现出来，我们开始认识到，任何一件民艺作品都不是孤立的，它是民间生活中的一件具体的用品，对它们的研究实质上就是对老百姓的传统生活、传统文化进行研究。
主持人：对民间艺术的保护，实质上是对传统文化的保护吗?
潘教授：的确是这样。民间文化如同一种生态环境，在这种生态环境中生长了不同的文化果实，如果仅仅研究文化产品，就会淡化它内在的生命力。我们提出这样的命题，目的就是呼吁大家要关爱与保护整个民间文化的生态环境。
主持人：我想保护民间文化不应该是重新回到传统里去吧?
潘教授：不是回到传统中去，而是反思传统，为了使我们的民族、我们的国家更快、更好也更加稳定地向前发展。传统文化是现代文化的根基，现代文化是传统文化的生命力。一个国家和民族，要生存就要发展，但发展离不开一个稳定的根基。
主持人：您刚才提到民间艺术就是传统的物质文化，我们现在应该如何保护?
潘教授：怎样把这些传统文化方式留下来，我觉得有一个重要的渠道，就是通过基础教育，通过我们的中学生、小学生。他们从小就需要这种体验，假如说让这些孩子从小学习编制一下中国结，做一些手工织绣的东西，我感觉是非常有益的。如果把这些东西跟基础教育、跟我们的这种文化产业结合起来，传统文化将会更有生命力。
Question: 提出这个命题的目的是什么?
Option: 1. 进行文化科学研究 2. 更好地发展传统文化 3. 全面地保护民间文化 4. 保护民间文化的生态环境
Answer: 4
Predicted Answer: 3
------------------------------


100%|██████████████████████████████████████████████████████████████████████████████| 1890/1890 [40:44<00:00,  1.29s/it]


處理完成！
總題數: 1890
答對題數: 1765
正確率 (Accuracy): 93.39%
詳細結果已儲存至 result_analysis.xlsx
